# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and basic processing of the FAIR² dataset published as Croissant metadata using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata  # Keep as an object, not a dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and column `@id`s.

In [ ]:
# List available RecordSets, their fields, and columns (@id usage everywhere)

print("Available RecordSets in the dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}  | Name: {rs['name']}\n    Columns: ")
    for field in rs['fields']:
        print(f"      - Field @id: {field['@id']} | Column/label: {field.get('name','')} | DataType: {field.get('dataType', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the `@id` field.

In [ ]:
# Get all record set IDs
record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id} with shape {df.shape}")

# Pick the main data record set for demonstration (usually first, or by known @id if dataset structure is known)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in RecordSet {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Choose numeric fields for simple filtering, normalization, and grouping.
Replace the example field IDs with those found in Section 2.

In [ ]:
# Example EDA: Change these to IDs present in your dataset
# You can find the relevant field @id from step 2: e.g., '@id': 'field:MW_kDa' for molecular weight

# Replace with actual field IDs used in your dataset for analysis:
record_set_id = main_record_set_id
# Example: 'field:MW_kDa' and 'field:Gene_Symbol'
numeric_field_id = None
group_field_id = None

if record_set_id:
    columns = dataframes[record_set_id].columns.tolist()
    print("Available columns:", columns)
    # Try to infer MW (molecular weight) or an abundance field
    for col in columns:
        if 'MW' in col or 'weight' in col:
            numeric_field_id = col
        if 'Gene' in col or 'Protein' in col or 'symbol' in col:
            group_field_id = col
    
    if numeric_field_id is None:
        numeric_field_id = columns[0]  # Fallback
    if group_field_id is None:
        group_field_id = columns[1] if len(columns)>1 else columns[0]
    print(f"\nUsing numeric_field_id: {numeric_field_id}")
    print(f"Using group_field_id: {group_field_id}\n")
    # Drop rows with missing values in the numeric field
    df = dataframes[record_set_id].copy()
    df = df[df[numeric_field_id].notnull()]
    # Convert to numeric if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df = df[df[numeric_field_id].notnull()]
    threshold = df[numeric_field_id].quantile(0.75)  # Top 25% values
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No record sets/columns for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id in dataframes[record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id in dataframes[record_set_id].columns:
        # Show top 10 groups by mean value
        grouped = dataframes[record_set_id].groupby(group_field_id)[numeric_field_id].mean().reset_index()
        grouped = grouped.sort_values(numeric_field_id, ascending=False).head(10)
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped[group_field_id], y=grouped[numeric_field_id])
        plt.title(f'Top 10 {group_field_id} by mean {numeric_field_id}')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and interact with the FAIR² dataset's Croissant metadata and tabular data using `mlcroissant`. We reviewed record sets and fields by their `@id`, loaded and explored the main record set in a pandas DataFrame, applied basic EDA, and visualized key patterns in the data.

You can now proceed to more specific analyses by referencing fields and record sets with their `@id`.